In [136]:
from data import data,affine,hardi_img,vox_size,labels,bvals,bvecs
from env_main import UnifiedTractographyEnv
from agent import PeakFollowingAgent,RandomTractographyAgent
import numpy as np
from fury import window, actor

In [ ]:
env = UnifiedTractographyEnv(
    data, affine, labels, bvals, bvecs, vox_size
    fa_threshold=0.25,      # Stopping criteria 1
    max_curvature_deg=45,   # Stopping criteria 2
    max_steps=500           # Stopping criteria 3
)

c:\Users\Samsung\Desktop\tractography\main\env_main.py:65: UserWarning: Pass ['bvecs'] as keyword args. From version 2.0.0 passing these as positional arguments will result in an error. 
  gtab = gradient_table(bvals, bvecs)


In [18]:
scene = window.Scene()
env.render_seeds_mask(scene)
env.render_wm_surface(scene, opacity=0.1)

<Scene(0x000001225B24D120) at 0x000001226CC28940>

In [54]:
window.show(scene,size=(800,800))

In [20]:
"""Visualizes all potential starting points (label 2)."""
coords = np.argwhere(labels == 2)
# Convert voxel seeds to world space for rendering
hom_coords = np.c_[coords, np.ones(len(coords))]
world_coords = (hom_coords @ affine.T)[:, :3]

seed_points = actor.line(
    lines=world_coords,
    colors=(0,1,0),
    

)

scene.add(seed_points)

In [42]:
scene.clear()

In [30]:
bvecs[40,:],bvals[40]

(array([ 0.74913,  0.45794, -0.47863]), 2000.0)

In [31]:
coords = np.argwhere((labels == 2)|(labels ==1))

In [33]:
# Apply affine to get world coordinates
hom_coords = np.c_[coords, np.ones(len(coords))]
world_coords = hom_coords @ affine.T
points = world_coords[:, :3]

In [32]:
csa_peaks=env.peaks
peak_dirs = csa_peaks.peak_dirs[coords[:,0], coords[:,1], coords[:,2]]  # (N,5,3)
peak_vals = csa_peaks.peak_values[coords[:,0], coords[:,1], coords[:,2]]  # (N,5)

In [52]:
from matplotlib import cm

lines = []
colors = []

for p, dirs, vals in zip(points, peak_dirs, peak_vals):
    for d, v in zip(dirs, vals):
        if v > 0:  # valid peak
            start = p - d / 2   # scale for visibility
            end   = p + d / 2
            lines.append([start, end])
            c = cm.viridis(v)[:3]  # RGB from colormap
            colors.append(c)


In [53]:
line_actor = actor.line(lines, colors=colors)
scene.add(line_actor)

In [62]:
from dipy.tracking import utils

In [83]:
seed_mask=(labels==2)
seeds= utils.seeds_from_mask(seed_mask, affine, density=vox_size)



In [81]:
sm=actor.point(
    points=seeds,
    colors=(1,0,0)
)
scene.add(sm)

In [95]:
window.show(scene=scene)

In [67]:
seed_mask.shape

(81, 106, 76)

In [66]:
seeds_1=np.argwhere(seed_mask)
seeds_1.shape

(505, 3)

In [74]:
hom_c=np.c_[seeds_1,np.ones(len(seeds_1))]
world_c= hom_c @ affine.T
world_c[:,:3]

array([[ -4., -44.,   8.],
       [ -4., -38.,   6.],
       [ -4., -36.,   6.],
       ...,
       [  0.,  32.,   8.],
       [  0.,  32.,  10.],
       [  0.,  32.,  12.]])

In [ ]:
from dipy.tracking.tracker import eudx_tracking
from dipy.tracking.stopping_criterion import ThresholdStoppingCriterion
stopping_criterion = ThresholdStoppingCriterion(csa_peaks.gfa, 0.25)
streamlines_generator = eudx_tracking(
    seeds, stopping_criterion, affine, step_size=0.5, pam=csa_peaks
)
from dipy.tracking.streamline import Streamlines
streamlines = Streamlines(streamlines_generator)   # consumes generator, returns list-like container
len(streamlines)


In [94]:
print(streamlines[32])

[[ -4.5        -36.5          5.5       ]
 [ -4.01867131 -36.37729996   5.55716113]
 [ -3.53740614 -36.24800771   5.59799436]
 [ -3.05567445 -36.12036685   5.63852075]
 [ -2.57394718 -35.99270836   5.67904412]
 [ -2.09270964 -35.86331494   5.71988249]
 [ -1.61240605 -35.73336946   5.76910273]
 [ -1.12680468 -35.62517518   5.81895559]
 [ -0.6352021  -35.54852324   5.86846709]
 [ -0.13930653 -35.506278     5.91645585]
 [  0.35868998 -35.49949823   5.96065441]]


In [97]:
scene_1=window.Scene()

In [ ]:
from dipy.viz import colormap
streamlines_actor= actor.line(
    streamlines, colors=colormap.line_colors(streamlines)
)

In [105]:
scene.rm(streamlines_actor)

In [111]:
window.show(scene,size=(800,800))

In [103]:

def custom_tracker(seeds, csa_peaks, affine, step_size=0.5, gfa_thresh=0.25, max_steps=500):
    """
    Manual implementation of a deterministic peak-following tracker.
    """
    all_streamlines = []
    gfa = csa_peaks.gfa
    peaks = csa_peaks.peak_dirs  # Shape (X, Y, Z, 5, 3)
    
    # Invert affine to move from world coordinates (seeds) to voxel indices
    inv_affine = np.linalg.inv(affine)

    for seed in seeds:
        # Convert seed to voxel space
        current_pos_world = np.array(seed)
        current_pos_vox = (np.append(seed, 1) @ inv_affine.T)[:3]
        
        streamline = [current_pos_world.copy()]
        prev_dir = None
        
        for _ in range(max_steps):
            # 1. Get voxel index
            idx = tuple(np.floor(current_pos_vox).astype(int))
            
            # 2. Stopping Criteria
            # Check boundaries
            if any(i < 0 or i >= s for i, s in zip(idx, gfa.shape)):
                break
            # Check GFA
            if gfa[idx] < gfa_thresh:
                break
                
            # 3. Get local peaks
            local_peaks = peaks[idx]  # Usually 5 peaks per voxel
            
            # 4. Determine direction
            if prev_dir is None:
                # First step: use the primary peak (peak 0)
                best_dir = local_peaks[0]
            else:
                # Follow-up steps: pick peak most aligned with prev_dir
                # Use absolute dot product because fiber orientations are axial
                dots = np.dot(local_peaks, prev_dir)
                best_peak_idx = np.argmax(np.abs(dots))
                best_dir = local_peaks[best_peak_idx]
                
                # Ensure we don't go backwards (keep angle < 90 deg)
                if np.dot(best_dir, prev_dir) < 0:
                    best_dir = -best_dir
                
                # Curvature check: angle > 45 degrees
                cos_sim = np.dot(best_dir, prev_dir)
                if cos_sim < np.cos(np.deg2rad(45)):
                    break

            # 5. Move in World Space (EuDX logic)
            # EuDX moves in the direction of the peak, scaled by step_size
            # We must convert the direction to world space for the next world point
            # or keep it in voxel space and step there.
            
            # Step in voxel space
            current_pos_vox += best_dir * (step_size / 1.0) # Assumes 1mm isotropic or handles via affine
            
            # Convert new voxel pos to world pos for the streamline list
            new_pos_world = (np.append(current_pos_vox, 1) @ affine.T)[:3]
            streamline.append(new_pos_world)
            
            prev_dir = best_dir

        if len(streamline) > 1:
            all_streamlines.append(np.array(streamline))
            
    return all_streamlines

# Usage:
my_streamlines = custom_tracker(seeds, csa_peaks, affine)

In [104]:
len(my_streamlines)

3324

In [109]:
my_streamlines_actor= actor.line(
    my_streamlines, colors=colormap.line_colors(streamlines)
)

In [110]:
scene.add(my_streamlines_actor)

In [128]:

scene_1.add(streamlines_actor)
window.show(scene_1)
window.show(scene)

In [117]:
# Start with identity affine (no rotation, no scaling, no translation)
affine_shift = np.eye(4)

# Add translation of +50 along x-axis
affine_shift[0, 3] = 50

print(affine_shift)

[[ 1.  0.  0. 50.]
 [ 0.  1.  0.  0.]
 [ 0.  0.  1.  0.]
 [ 0.  0.  0.  1.]]


In [127]:
shifted_streamlines = [sl + np.array([5000, 0, 0]) for sl in my_streamlines]

my_streamlines_actor_shifted= actor.line(
    my_streamlines, colors=colormap.line_colors(shifted_streamlines)

)

scene_1.add(my_streamlines_actor_shifted)
window.show(scene_1)

In [126]:
scene_1.rm(my_streamlines_actor_shifted)
window.show(scene_1)

In [125]:
scene_1.clear()